# Pattern 05: HyDE (Hypothetical Document Embeddings)

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both LLM calls (the
hypothetical-document generation and the final answer) and the embedding step. Two LLM calls per
question means real cost is roughly double a naive-dense query's -- see `recipes/hyde.py`'s token
accounting, which sums both calls (a bug found and fixed while designing this pattern: silently
using only the final call's tokens would understate cost by about half). Section 7 is a PENDING
placeholder awaiting a real-embeddings-and-LLM run.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

HyDE asks the LLM to write a short *hypothetical* passage that would answer the question --
plausible-sounding, even if factually invented -- then embeds and searches with **that** passage
instead of the raw question (`prompts/hyde_prompt.txt`). The intuition: a hypothetical answer is
often closer in embedding space to a real answer passage than the bare question is, since
questions and their answers are phrased very differently even when semantically related. The real
answer is generated afterward from whatever gets retrieved, using the held-constant generation
prompt as usual.


## 2. When to use it

- Questions are phrased very differently from how the answer appears in source documents (e.g. a
  conversational question vs. a formal paper abstract)
- You've already tried naive dense retrieval and it's missing relevant chunks that *are* in the
  corpus but phrased far from the question
- You can afford 2 LLM calls per query instead of 1


## 3. When NOT to use it

- Cost/latency budget only allows 1 LLM call per query -- see pattern 01 or 02 instead
- The hypothetical passage risks hallucinating specifics (exact numbers, identifiers) that then
  pull retrieval toward the *wrong* chunk that happens to share that invented specificity
- Queries are already close in vocabulary to the source documents (BM25 or naive dense may already
  work fine, with less cost)


## 4. Implementation

In [3]:
from recipes.hyde import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)
print("input_tokens (summed across both LLM calls):", sample.input_tokens)


retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00088#2', 'arxiv:2601.00088#0']
answer: PCEVAL stands for Physical Computing Evaluation. It is a benchmark designed for evaluating the physical computing capabilities of Large Language Models (LLMs), assessing their ability to generate circuits and produce compatible code in physical computing projects [arxiv:2601.02404#0].
input_tokens (summed across both LLM calls): 1643


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="05_hyde",
    judges_enabled=True,
)


=== 05_hyde (n=18) ===
  hit@3: 1.000  [95% CI 1.000, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.833  [95% CI 0.722, 0.944]
  faithfulness: 0.722  [95% CI 0.500, 0.944]
  answer_relevance: 0.889  [95% CI 0.722, 1.000]
  citation_accuracy: 0.815  [95% CI 0.667, 0.926]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 5966.7
  p95_latency_ms: 22381.8
  usd_per_query: $0.01013
  eval_usd: $0.1823


## 6. Example query walkthrough

One example per eval-set category, showing the retrieved chunks (found via the hypothetical
document, not the raw question) and the (mocked) final answer. Under mock, the "hypothetical
document" is just `MockLLM`'s canned text, not a real plausible passage -- so which chunks come
back is not meaningful, only that the two-call code path runs end to end.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00088#2', 'arxiv:2601.00088#0']
A: PCEVAL stands for Physical Computing Evaluation. It is a benchmark designed for evaluating the physical computing capabilities of Large Language Models (LLMs), assessing their ability to generate circuits and produce compatible code in physical computing projects [arxiv:2601.02404#0].



--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: Repeated image generations from the same text prompt in diffusion models end up looking very similar due to a phenomenon called mode collapse. This occurs because the models tend to produce nearly identical outputs across different random seeds when given a fixed prompt, limiting the diversity of generated images. The initial noise inputs play a crucial role in this, and without proper variation or optimization, the model repeatedly converges to similar outputs. This issue has been observed in contemporary text-to-image models like Stable Diffusion SDXL-Turbo and Flux.1, where sampling multiple images from the same prompt yields surprisingly little visual variation [arxiv:2601.00090#0].



--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00130#2', 'arxiv:2601.00129#0', 'arxiv:2601.00129#2']
A: The two photonics-AI-systems papers focus on complementary aspects of the challenge of realizing large-scale photonics-empowered AI systems:

1. The paper in [arxiv:2601.00129] primarily addresses the system-level integration challenges and the need for physical design automation and system-algorithm co-exploration. It emphasizes the importance of scalable, manufacturable implementation of complex photonic integrated circuits (PICs) and electronic-photonic integrated circuits (EPICs) through physical design automation (EPDA). It also highlights the necessity of incorporating physical non-idealities, control/calibration limits, and packaging/interface costs into architectural design and learning optimization to bridge the gap between device-level advances 

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00907#2', 'arxiv:2601.00907#0', 'arxiv:2601.00907#1']
A: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication using deep learning. It focuses on Placenta Accreta Spectrum (PAS), a life-threatening obstetric condition, and develops a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve early and accurate prenatal diagnosis. The study demonstrates that combining MRI and US features through deep learning models significantly enhances diagnostic performance compared to unimodal approaches [arxiv:2601.00907#0, #2].



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal HyDE retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.hyde import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
